# Example of inference to classify X-ray images in: normal/pathological

In [2]:
import cv2
import numpy as np
from PIL import Image

# AI framework
import torch
from torchvision import transforms

# Device
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f'Device: {DEVICE}')

Device: cuda


### Load model

In [3]:
model = torch.load("normal_resnet18_TL_1out.pth", map_location = torch.device(DEVICE))

### Preprocess the image

In [4]:
def load_file(filename, monochrome):  
    pixels = cv2.imread(filename, cv2.IMREAD_UNCHANGED) 
    if pixels is None:
        return None

    if monochrome == "MONOCHROME1":
        pixels = np.amax(pixels) - pixels

    pixels = pixels - np.min(pixels)
    pixels = pixels / np.max(pixels)
    pixels = (pixels * 255).astype(np.uint8)

    return pixels

def equalize(image):    
    clahe = cv2.createCLAHE(clipLimit=40.0, tileGridSize=(8,8))
    clahe_img = clahe.apply(image)

    kernel = cv2.getStructuringElement(cv2.MORPH_RECT, (15, 15)) # MORPH_ELLIPSE
    eq_img = cv2.equalizeHist(image)
    tophat = cv2.morphologyEx(eq_img, cv2.MORPH_TOPHAT, kernel)
    bothat = cv2.morphologyEx(eq_img, cv2.MORPH_BLACKHAT, kernel)
    hat_img = eq_img + tophat - bothat

    return np.dstack((image, clahe_img, hat_img))

In [5]:
image_transform = transforms.Compose([
                            transforms.Resize((512,512)),
                            transforms.ToTensor()
                            ])

### Load an image and apply transforms

In [6]:
# Image path
img_path = "../Data/PC_512/100014625199913409730274754282179594842_0jycky.png"

# Step 1: Load the image
image_original = load_file(img_path, "MONOCHROME2")

# Step 2: Equalize
image_eq = equalize(image_original)
image_eq = image_eq[...,::-1] # Invert the order of the channels

# Step 3: Resize and transform to tensor   
img_trans = image_transform(Image.fromarray(image_eq)).unsqueeze(0).to(DEVICE)

### Run the inference

In [7]:
model.eval()

# Forward
outputs = model(img_trans)

# Obtain the probabilities
score = torch.sigmoid(outputs)
score_np = score.data.cpu().numpy()

# Use a threshold of 0.5 (default)
preds = score > 0.5

In [11]:
# Check if the prediction is pathological or normal
if preds:
    prediction = "pathological"
    probability = score.item()*100
else:
    prediction = "normal"
    probability = (1 - score.item())*100

# Print the message with the prediction and probability
print(f"The image is {prediction} with a probability of {probability:.2f}")


The image is normal with a probability of 69.87
